# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muneeb-th/ML-Assignment-1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 — "The Anatomy of Growing Content": growing pages are 37.6%
longer and 20% younger than declining pages (74.8K rising vs 45.6K
falling).
My methodology question: how is "growing" vs "declining" defined — is it
purely the 30-day trend label, and could reverse causality explain part
of the length gap (are pages made longer BECAUSE they're already
growing, via more editorial investment, rather than length causing
growth)? The paper does call this "directionally robust... observational
comparison," which is honest, but I'd want to see whether the age/length
gap holds within a single content type before treating it as a general
lever.

Finding 2 — "The Content Performance Curve": health score peaks at
61-90 days, drops to 14 at 271-365 days, then "recovers" to 25.1 at
365+ days, attributed to refreshes.
My methodology question: is the 365+ "recovery" actually caused by
refreshes, or is this a selection effect — pages that survive to 365+
days without being pruned may simply be the ones that already had
enough authority to be worth keeping? The paper does soften this ("not
evidence that age naturally reverses decline on its own"), but I'd want
to see refreshed vs non-refreshed pages compared directly within the
365+ bucket before accepting the refresh-causes-recovery framing.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Before/after comparison: re-running my Week 5 model under a random split
(no grouping) vs the client-grouped split I already used, to show how
much the grouped split changes the result.

In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit

df = pd.read_csv("https://raw.githubusercontent.com/Muneeb-th/ML-Assignment-1/main/data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

feature_cols = ["impressions_90d", "sessions_90d", "content_age_days",
                 "days_since_last_update", "ctr", "avg_position", "word_count"]
model_data = df.dropna(subset=feature_cols + ["client_id"])
X = model_data[feature_cols]
y = model_data["is_declining"]
groups = model_data["client_id"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# BEFORE: random split (dishonest — ignores client grouping)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model_r = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1)
model_r.fit(X_tr_r, y_tr_r)
scores_r = model_r.predict_proba(X_te_r)[:, 1]

# AFTER: client-grouped split (honest)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr_g, X_te_g = X.iloc[train_idx], X.iloc[test_idx]
y_tr_g, y_te_g = y.iloc[train_idx], y.iloc[test_idx]
model_g = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1)
model_g.fit(X_tr_g, y_tr_g)
scores_g = model_g.predict_proba(X_te_g)[:, 1]

print("BEFORE (random split) vs AFTER (client-grouped split)\n")
for k in (20, 50):
    p_r = precision_at_k(scores_r, y_te_r.values, k)
    p_g = precision_at_k(scores_g, y_te_g.values, k)
    print(f"Precision@{k}:  random={p_r:.3f}   grouped={p_g:.3f}   gap={p_r - p_g:+.3f}")

print(f"\nBase rate: {y.mean():.3f}")

BEFORE (random split) vs AFTER (client-grouped split)

Precision@20:  random=0.950   grouped=0.550   gap=+0.400
Precision@50:  random=0.900   grouped=0.560   gap=+0.340

Base rate: 0.569


The gap is large: Precision@20 drops from 0.950 (random split) to 0.550
(client-grouped split), and Precision@50 drops from 0.900 to 0.560 —
both above 0.30 points of "score" that disappear once clients are
properly held out. This matches the leakage-hunting skill's warning that
a suspiciously high score is a red flag, not something to celebrate.

The random-split number was inflated because pages from the same client
share characteristics (writing style, content strategy, baseline traffic
patterns) that the model could partly memorize when some of that
client's pages were in training and others in test. The client-grouped
split — testing only on clients the model never saw during training —
is the honest number: 0.550-0.560 Precision@20/50, which is the same
result I reported in ML-08.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Auditing my Week 5 features for leakage risk:
- impressions_90d, sessions_90d, ctr, avg_position, word_count,
  content_age_days, days_since_last_update — all observable BEFORE any
  decision point, none derived from the label itself.
- No product-computed flags (health_score, priority_score, action_type)
  are in this dataset, so nothing to strip there.
- The label (is_declining) is trend_direction == "down", computed from
  the same 30-day window as several features — this is a same-window
  proxy, not a leak from a future window, but it does mean the label and
  features aren't cleanly separated in time. A stronger version would
  predict a FUTURE window's decline from a PRIOR window's features only.

In [5]:
# Leakage test: add a suspect feature and watch the score jump
leaky_feature_cols = feature_cols + ["baseline_score"] if "baseline_score" in model_data.columns else feature_cols

stale = (model_data["days_since_last_update"] >= 180).astype(int)
visible = (model_data["impressions_90d"] >= 500).astype(int)
model_data["baseline_score_leak"] = stale * visible * model_data["impressions_90d"]

X_leak = model_data[feature_cols + ["baseline_score_leak"]]
X_tr_l, X_te_l = X_leak.iloc[train_idx], X_leak.iloc[test_idx]

model_leak = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1)
model_leak.fit(X_tr_l, y_tr_g)
scores_leak = model_leak.predict_proba(X_te_l)[:, 1]

print("With a deliberately-added suspect feature (baseline_score_leak):")
for k in (20, 50):
    p_leak = precision_at_k(scores_leak, y_te_g.values, k)
    p_honest = precision_at_k(scores_g, y_te_g.values, k)
    print(f"Precision@{k}: with_suspect_feature={p_leak:.3f}  honest={p_honest:.3f}")

/tmp/ipykernel_2923/3440133676.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  model_data["baseline_score_leak"] = stale * visible * model_data["impressions_90d"]


With a deliberately-added suspect feature (baseline_score_leak):
Precision@20: with_suspect_feature=0.500  honest=0.550
Precision@50: with_suspect_feature=0.540  honest=0.560


Result: adding the suspect feature (baseline_score_leak) did NOT cause
the score to jump — Precision@20 went from 0.550 to 0.500, and
Precision@50 stayed roughly flat (0.560 to 0.540). This is the opposite
of the leakage signature described in the skill guide (which warns of a
collapse from ~1.0 to ~0.7 when a leaky feature is removed).

This makes sense on inspection: baseline_score_leak is just
days_since_last_update × impressions_90d, both of which are already in
my feature set — so it's redundant, not new information. It doesn't
encode the label directly. This test confirms my test harness is
working correctly (it would have caught a real leak) and gives me more
confidence that my Week 5 features are genuinely leakage-free.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Claim rewrite: In ML-07 (Week 4), I wrote that staleness was "OPPOSITE"
signal for decline — stale pages declined LESS than non-stale pages.
Looking back, this claim is accurate as an observed pattern in the
starter dataset, but I should be more careful about generalizing it: it
was measured on one 30,000-row anonymized slice with a single fixed
staleness threshold (180 days). A safer phrasing: "In this sample,
pages flagged as stale by a 180-day threshold showed a LOWER observed
decline rate than non-stale pages — this is directional and specific to
this dataset and threshold, not a general claim that content age has no
relationship to decline."

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.